# 1. Project and preprocessing objective

Prepare one reproducible single-class dataset for detecting the complete visible UAE license plate. This notebook explains the shared scripts; it does not replace them or train a model.

In [ ]:
TARGET_CLASS = {"id": 0, "name": "license_plate"}
print(TARGET_CLASS)

## 2. Dataset source and CC BY 4.0 attribution

The handoff attributes the UAE dataset to the required course source URL under CC BY 4.0. The local raw export metadata discrepancy is kept as a limitation.

In [ ]:
SOURCE_URL = "https://universe.roboflow.com/addinguae/uae-zcfqj"
SOURCE_LICENSE = "CC BY 4.0"
print(SOURCE_URL, SOURCE_LICENSE, sep="\n")

## 3. AI assistance acknowledgment

Generative AI tools, including ChatGPT and Codex, were used to assist with portions of the preprocessing code structure, debugging, and documentation. Dataset statistics and validation statements in this repository must be produced by running the included scripts against the stated dataset release. Human-only review fields must remain clearly identified and must not be completed automatically.

In [ ]:
AI_ASSISTANCE_ACKNOWLEDGED = True
print("Human-only review remains incomplete.")

## 4. Dataset paths

Paths are discovered from the notebook working directory, so no manual path editing is needed when this handoff retains its documented layout.

In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
candidates = [start, *start.parents]
PACKAGE_ROOT = next(path for path in candidates if (path / "scripts" / "preprocessing_utils.py").is_file())
DATASET_ROOT = PACKAGE_ROOT / "datasets" / "uae_lp_v2_yolo"
SOURCE_ROOT = PACKAGE_ROOT.parent / "datasets" / "raw_v2" / "addinguae_roboflow"
sys.path.insert(0, str(PACKAGE_ROOT / "scripts"))
assert DATASET_ROOT.is_dir() and SOURCE_ROOT.is_dir()
print(PACKAGE_ROOT, DATASET_ROOT, SOURCE_ROOT, sep="\n")

## 5. Original source class names and IDs

Class names are read from the actual original `data.yaml`, not copied from documentation.

In [ ]:
from preprocessing_utils import read_class_names

source_names = read_class_names(SOURCE_ROOT / "data.yaml")
assert len(source_names) == 51
for class_id, class_name in enumerate(source_names):
    print(f"{class_id:2d}: {class_name}")

## 6. Original class distribution

`class_mapping.csv` is generated by parsing the actual source label rows.

In [ ]:
import csv

with (PACKAGE_ROOT / "reports" / "class_mapping.csv").open(encoding="utf-8", newline="") as handle:
    class_mapping = list(csv.DictReader(handle))
assert len(class_mapping) == len(source_names)
for row in class_mapping:
    print(row["source_class_id"], row["source_class_name"], row["source_box_count"])

## 7. Source-to-target class mapping

Only the exact source class `plate` represents the full visible target. All other source classes are non-target OCR, marks, emirate, or style annotations.

In [ ]:
kept = [row for row in class_mapping if row["decision"] == "KEEP_AND_MAP"]
removed = [row for row in class_mapping if row["decision"] == "REMOVE_NON_TARGET"]
assert len(kept) == 1 and kept[0]["source_class_name"] == "plate"
print("Kept:", kept[0])
print("Removed non-target boxes:", sum(int(row["boxes_removed"]) for row in removed))

## 8. Original and final image, label and box counts

The audit distinguishes observed values from unavailable historical claims.

In [ ]:
with (PACKAGE_ROOT / "reports" / "preprocessing_audit.csv").open(encoding="utf-8", newline="") as handle:
    audit_rows = list(csv.DictReader(handle))
for row in audit_rows:
    print(f"{row['metric']}: before={row['before']}, after={row['after']}")

## 9. Train, validation and test split counts

The accepted release has three non-empty, separate splits.

In [ ]:
import json

release = json.loads((PACKAGE_ROOT / "dataset_release.json").read_text(encoding="utf-8"))
for split in ("train", "val", "test", "total"):
    print(split, release["split_counts"][split], release["box_counts"][split])

## 10. Explanation of why validation and test remain separate

Validation supports model selection and tuning. Test is reserved for final unbiased evaluation. Combining them would leak evaluation information into development decisions.

In [ ]:
assert release["split_counts"]["val"] > 0
assert release["split_counts"]["test"] > 0
print("Validation and test remain separate in both YOLO and COCO.")

## 11. Random ground-truth bounding-box samples

The shared visualization script uses fixed seed 486 and draws every ground-truth box.

In [ ]:
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

def show_images(paths, titles):
    figure, axes = plt.subplots(1, len(paths), figsize=(6 * len(paths), 5))
    axes = np.atleast_1d(axes)
    for axis, path, title in zip(axes, paths, titles):
        image = cv2.imread(str(path), cv2.IMREAD_COLOR)
        assert image is not None, path
        axis.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        axis.set_title(title)
        axis.axis("off")
    plt.tight_layout()
    plt.show()

show_images([PACKAGE_ROOT / "reports" / "figures" / f"{split}_random_samples.jpg" for split in ("train", "val", "test")], ["Train", "Validation", "Test"])

## 12. Smallest-box examples

Selection is deterministic from measured normalized box area.

In [ ]:
show_images([PACKAGE_ROOT / "reports" / "figures" / "smallest_boxes.jpg"], ["Smallest boxes"])

## 13. Largest-box examples

Selection is deterministic from measured normalized box area.

In [ ]:
show_images([PACKAGE_ROOT / "reports" / "figures" / "largest_boxes.jpg"], ["Largest boxes"])

## 14. Multi-plate examples

Images are ordered by observed plate count and then filename.

In [ ]:
show_images([PACKAGE_ROOT / "reports" / "figures" / "multi_plate_images.jpg"], ["Multi-plate images"])

## 15. Edge-touching box examples

These samples are selected from boxes measured at an image boundary using the documented tolerance.

In [ ]:
show_images([PACKAGE_ROOT / "reports" / "figures" / "edge_touching_boxes.jpg"], ["Edge-touching boxes"])

## 16. Scene/template leakage examples

Historical review sheets show candidate scene/template leakage, where vehicle and background may be reused while plate text differs. Human review is still required.

In [ ]:
show_images([PACKAGE_ROOT / "reports" / "near_duplicate_review_pairs" / "pair_001.jpg"], ["Historical pair 001: NEEDS_HUMAN_REVIEW"])

## 17. Training-only augmentation policy

Random transformations are training-only. Validation and test receive no random augmentation. Normalization and deterministic resizing are model-specific. Geometric transforms update every box, and samples below 70% box visibility are rejected. Horizontal and vertical flips are disabled.

In [ ]:
from preprocessing_utils import load_augmentation_policy

policy = load_augmentation_policy(PACKAGE_ROOT / "configs" / "augmentation_policy.yaml")
assert policy["enabled_for_train"] and not policy["enabled_for_validation"] and not policy["enabled_for_test"]
assert policy["horizontal_flip_probability"] == policy["vertical_flip_probability"] == 0.0
print(policy)

## 18. Augmentation preview with correctly transformed boxes

The preview is generated only from training images and is not a new split.

In [ ]:
show_images([PACKAGE_ROOT / "reports" / "figures" / "augmentation_preview.jpg"], ["Training-only augmentation preview"])

## 19. YOLO-to-COCO conversion example using one real annotation

The shared conversion function uses the actual decoded image dimensions.

In [ ]:
from preprocessing_utils import find_image_files, load_yolo_labels, read_image_size, yolo_to_coco_bbox

example_image = find_image_files(DATASET_ROOT / "images" / "train")[0]
example_label = DATASET_ROOT / "labels" / "train" / f"{example_image.stem}.txt"
example_box = load_yolo_labels(example_label, allow_empty=False)[0]
width, height = read_image_size(example_image)
computed_bbox = yolo_to_coco_bbox(example_box, width, height)
coco_train = json.loads((PACKAGE_ROOT / "annotations" / "coco" / "train.json").read_text(encoding="utf-8"))
coco_image = next(item for item in coco_train["images"] if item["file_name"].endswith(example_image.name))
coco_annotation = next(item for item in coco_train["annotations"] if item["image_id"] == coco_image["id"])
assert np.allclose(computed_bbox, coco_annotation["bbox"], atol=0.01)
print("YOLO:", example_box)
print("Computed COCO bbox:", computed_bbox)
print("Stored COCO bbox:", coco_annotation["bbox"])

## 20. Final validation results

Validation statements below are produced by calling the shared validator against the stated release.

In [ ]:
from validate_dataset import run_validation

validation = run_validation(PACKAGE_ROOT, DATASET_ROOT, "notebook shared run_validation", write_report=False)
assert validation["status"] == "PASS", validation["errors"][:10]
print(validation["status"], validation["counts"])

## 21. Known limitations

Human review is incomplete, five accepted-release omission reasons are not reconstructable, raw attribution metadata conflicts with the required URL, and the inspected training code does not yet implement the exact augmentation policy.

In [ ]:
for limitation in release["known_limitations"]:
    print("-", limitation)

## 22. Course learning outcome mapping

Three-way splitting maps to 11-TrainingCNN slides 38–39; augmentation to slide 35; normalization concepts to slides 5–6; full-object boxes to 13-Detection&Segmentation slides 27–33; YOLO to slide 40; brightness to 02-Image-formation slide 16; and Gaussian filtering to 03-Filtering slides 28–41.

Dataset statistics and visual inspection support experimentation and engineering judgment. SHA-256, perceptual hashing, Hamming distance, COCO conversion, and release manifests are project engineering tools supporting CLO 6. They are not presented as techniques taught directly in the lecture slides. SIFT, feature matching, edge detection, multiview geometry, optical flow, and tracking are not added merely to claim course coverage.

In [ ]:
engineering_tools = ["SHA-256", "perceptual hashing", "Hamming distance", "COCO conversion", "release manifests"]
print("CLO 6 supporting project engineering tools:", ", ".join(engineering_tools))